# Bronze → Silver: medidores

**Entrada:** `venergy.bronze.medidores` (texto bruto, 51.812 linhas).
**Saída:** `venergy.silver.medidores` (Delta, tipada e padronizada).
**Grão:** uma linha por medidor (`id_medidor` único). Os medidores retirados continuam na tabela, porque as leituras antigas apontam para eles.

**O que este notebook faz.** Cada passo corresponde a um achado de `docs/auditoria_dados/auditoria_medidores.md`:
1. **Padroniza o texto:** `trim` e maiúsculas em ids, `modelo` e `status`. `tipo` é reduzido ao domínio `ELETRONICO` / `ELETROMECANICO` / `SMART`.
2. **Converte as datas pelo formato de cada valor:** ISO, `dd/MM/yyyy`, `dd.MM.yyyy` e **`MM-dd-yyyy` para as datas com hífen**. A sentinela `9999-12-31` vira nulo.
3. **Remove duplicatas exatas** depois da padronização (513 linhas).
4. **Enriquece cada medidor com o anterior da mesma UC** (`id_medidor_anterior`, `tipo_anterior`), deixando a campanha de troca pronta para a Gold.
5. **Valida antes de gravar:** se uma regra crítica falhar, o notebook para e a Silver atual não é tocada. Carga completa com `overwrite` (idempotente).

Documentação: `docs/silver/silver_medidores.md`.

In [ ]:
catalog = "venergy"
bronze_schema = "bronze"
silver_schema = "silver"

bronze_table = f"{catalog}.{bronze_schema}.medidores"
silver_table = f"{catalog}.{silver_schema}.medidores"

COLUNAS_BRONZE = [
    "id_medidor", "id_uc", "modelo", "tipo", "data_instalacao",
    "data_retirada", "data_ultima_manutencao", "status",
]
TIPOS_VALIDOS = ["ELETRONICO", "ELETROMECANICO", "SMART"]
STATUS_VALIDOS = ["ATIVO", "RETIRADO"]
SENTINELAS_DATA = ["9999-12-31"]  # "sem data" na origem (medidor não retirado)

# Formatos de data encontrados na auditoria (seção 3.1). Os padrões são mutuamente exclusivos.
# Hífen = padrão AMERICANO: nas 1.263 datas com hífen o 1º campo nunca passa de 12.
FORMATOS_DATA = [
    (r"^\d{4}-\d{2}-\d{2}$", "yyyy-MM-dd"),
    (r"^\d{2}/\d{2}/\d{4}$", "dd/MM/yyyy"),
    (r"^\d{2}\.\d{2}\.\d{4}$", "dd.MM.yyyy"),
    (r"^\d{2}-\d{2}-\d{4}$", "MM-dd-yyyy"),
]

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")

## 1. Leitura da Bronze
Confere se as colunas esperadas existem antes de qualquer transformação.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

df_bronze = spark.table(bronze_table)
faltando = set(COLUNAS_BRONZE) - set(df_bronze.columns)
assert not faltando, f"Colunas ausentes na Bronze: {faltando}"

linhas_bronze = df_bronze.count()
print(f"{bronze_table}: {linhas_bronze} linhas")

## 2. Padronização, conversão de datas e deduplicação
- `texto()`: `trim` + maiúsculas.
- `tipo`: remove acento, tudo que não é letra e o sufixo `METER` (`eletrônico` → `ELETRONICO`, `ELETRO-MECANICO` → `ELETROMECANICO`, `smart meter` → `SMART`). A validação garante que o resultado cai no domínio.
- `para_data()`: escolhe o formato pelo padrão do próprio valor. Valor fora dos formatos conhecidos, ou data impossível (ex.: 30/02), vira nulo **e é barrado na validação**, sem quebrar a execução.
- As duplicatas são removidas **depois** da padronização, então duas linhas que só diferem por espaço ou formato de data também contam como a mesma.

In [ ]:
def texto(coluna):
    """Remove espaços nas pontas e converte para maiúsculas."""
    return F.upper(F.trim(F.col(coluna)))


def sem_acento(col):
    return F.translate(col, "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ", "AAAAAEEEEIIIIOOOOOUUUUC")


def para_data(coluna):
    """Converte texto em DATE escolhendo o formato pelo padrão do valor (ver FORMATOS_DATA)."""
    valor = F.trim(F.col(coluna))
    valor = F.when((valor == "") | valor.isin(SENTINELAS_DATA), F.lit(None)).otherwise(valor)
    data = None
    for regex, formato in FORMATOS_DATA:
        convertida = F.to_date(F.try_to_timestamp(valor, F.lit(formato)))
        data = F.when(valor.rlike(regex), convertida) if data is None else data.when(valor.rlike(regex), convertida)
    return data


def informada_sem_conversao(coluna):
    """Valor preenchido na Bronze (que não é sentinela) mas que não virou data."""
    bruto = F.trim(F.col(f"_{coluna}_bruta"))
    return bruto.isNotNull() & (bruto != "") & ~bruto.isin(SENTINELAS_DATA) & F.col(coluna).isNull()


tipo_padronizado = F.regexp_replace(
    F.regexp_replace(sem_acento(texto("tipo")), "[^A-Z]", ""), "METER$", ""
)

df_padronizado = df_bronze.select(
    texto("id_medidor").alias("id_medidor"),
    texto("id_uc").alias("id_uc"),
    texto("modelo").alias("modelo"),
    tipo_padronizado.alias("tipo"),
    texto("status").alias("status"),
    para_data("data_instalacao").alias("data_instalacao"),
    para_data("data_retirada").alias("data_retirada"),
    para_data("data_ultima_manutencao").alias("data_ultima_manutencao"),
    # valores originais das datas: usados só na validação, não vão para a Silver
    F.col("data_instalacao").alias("_data_instalacao_bruta"),
    F.col("data_retirada").alias("_data_retirada_bruta"),
    F.col("data_ultima_manutencao").alias("_data_ultima_manutencao_bruta"),
    "source_file",
    "ingestion_datetime",
)

COLUNAS_NEGOCIO = [
    "id_medidor", "id_uc", "modelo", "tipo", "status",
    "data_instalacao", "data_retirada", "data_ultima_manutencao",
]
df_unico = df_padronizado.dropDuplicates(COLUNAS_NEGOCIO)

## 3. Enriquecimento: medidor anterior da UC
Ordena os medidores de cada UC pela data de instalação. Para cada medidor, guarda o anterior e o tipo dele. Na extração atual, isso marca as 6.299 trocas da campanha de set/2025 a mar/2026 (5.661 eletromecânicos, 601 eletrônicos e 37 smart substituídos por smart).

A validação exige que o medidor anterior esteja `RETIRADO`. Se uma carga trouxer o `ATIVO` com instalação anterior à do retirado, a troca sairia invertida, então a carga para.

In [ ]:
por_uc = Window.partitionBy("id_uc").orderBy("data_instalacao", "id_medidor")

df_tratado = (
    df_unico
    .withColumn("id_medidor_anterior", F.lag("id_medidor").over(por_uc))
    .withColumn("tipo_anterior", F.lag("tipo").over(por_uc))
    .withColumn("_retirada_do_anterior", F.lag("data_retirada").over(por_uc))
    .withColumn("_status_do_anterior", F.lag("status").over(por_uc))
    .withColumn("silver_datetime", F.current_timestamp())
)

## 4. Validação antes de gravar
Cada regra conta as linhas que a **violam**, e todas são calculadas numa única passada pelos dados.
- **crítica:** qualquer violação interrompe o notebook antes do `overwrite`, e a Silver atual não é tocada. Antes de parar, o notebook mostra até 20 linhas que violam cada regra.
- **aviso:** é exibida para investigação, mas não bloqueia a carga.

In [ ]:
df_check = (
    df_tratado
    .withColumn("_repeticoes_id", F.count("*").over(Window.partitionBy("id_medidor")))
    .withColumn("_ativos_na_uc", F.sum((F.col("status") == "ATIVO").cast("int")).over(Window.partitionBy("id_uc")))
)

REGRAS = [  # (regra, condição que identifica uma linha que VIOLA a regra, crítica?)
    ("id_medidor repetido após remover duplicatas exatas", F.col("_repeticoes_id") > 1, True),
    ("id_medidor ou id_uc fora do padrão MD…/UC… + dígitos",
     F.col("id_medidor").isNull() | ~F.col("id_medidor").rlike(r"^MD\d+$")
     | F.col("id_uc").isNull() | ~F.col("id_uc").rlike(r"^UC\d+$"), True),
    ("modelo fora do padrão AA-999", F.col("modelo").isNull() | ~F.col("modelo").rlike(r"^[A-Z]{2}-\d{3,4}$"), True),
    ("tipo fora do domínio", F.col("tipo").isNull() | ~F.col("tipo").isin(TIPOS_VALIDOS), True),
    ("status fora do domínio", F.col("status").isNull() | ~F.col("status").isin(STATUS_VALIDOS), True),
    ("data_instalacao vazia ou não convertida", F.col("data_instalacao").isNull(), True),
    ("data_retirada informada mas não convertida", informada_sem_conversao("data_retirada"), True),
    ("data_ultima_manutencao informada mas não convertida", informada_sem_conversao("data_ultima_manutencao"), True),
    ("status incoerente com data_retirada",
     ((F.col("status") == "RETIRADO") & F.col("data_retirada").isNull())
     | ((F.col("status") == "ATIVO") & F.col("data_retirada").isNotNull()), True),
    ("retirada anterior à instalação", F.col("data_retirada") < F.col("data_instalacao"), True),
    ("instalação no futuro", F.col("data_instalacao") > F.current_date(), True),
    ("UC sem exatamente 1 medidor ATIVO", F.col("_ativos_na_uc") != 1, True),
    ("medidor anterior não está RETIRADO (troca invertida)",
     F.col("id_medidor_anterior").isNotNull() & (F.col("_status_do_anterior") != "RETIRADO"), True),
    ("troca com lacuna: instalação diferente da retirada do anterior",
     F.col("id_medidor_anterior").isNotNull()
     & (F.col("_retirada_do_anterior").isNull() | (F.col("_retirada_do_anterior") != F.col("data_instalacao"))), False),
]

contagens = df_check.agg(
    F.count("*").alias("linhas"),
    *[F.count_if(condicao).alias(f"regra_{i}") for i, (_, condicao, _) in enumerate(REGRAS)],
).first()

linhas_silver = contagens["linhas"]
resultado = [
    (regra, int(contagens[f"regra_{i}"]), "crítica" if critica else "aviso")
    for i, (regra, _, critica) in enumerate(REGRAS)
]
display(spark.createDataFrame(
    [(r, v, n, "OK" if v == 0 else ("FALHA" if n == "crítica" else "ATENÇÃO")) for r, v, n in resultado],
    "regra string, violacoes long, nivel string, status string",
))
print(f"Bronze: {linhas_bronze} linhas | Silver: {linhas_silver} linhas | duplicatas removidas: {linhas_bronze - linhas_silver}")

falhas = [f"{r}: {v} linha(s)" for r, v, n in resultado if v and n == "crítica"]
for (regra, condicao, critica), (_, violacoes, _) in zip(REGRAS, resultado):
    if critica and violacoes:
        print(f"Exemplos de linhas que violam: {regra}")
        display(df_check.where(condicao).limit(20))
assert not falhas, "Silver de medidores reprovada na validação:\n- " + "\n- ".join(falhas)

## 5. Escrita na Silver
Os comentários de coluna vão no schema (`withMetadata`) e são gravados no mesmo commit da carga. O comentário da tabela só é reescrito quando muda, para o histórico Delta mostrar apenas as cargas.

In [ ]:
COMENTARIOS = {
    "id_medidor": "Identificador do medidor (chave da tabela).",
    "id_uc": "Unidade consumidora onde o medidor está ou esteve instalado.",
    "modelo": "Modelo do medidor, sem espaços (ex.: EL-3000).",
    "tipo": "Tecnologia do medidor: ELETRONICO, ELETROMECANICO ou SMART.",
    "status": "ATIVO (instalado hoje) ou RETIRADO.",
    "data_instalacao": "Data de instalação do medidor na UC.",
    "data_retirada": "Data de retirada; nula enquanto o medidor está ATIVO.",
    "data_ultima_manutencao": "Data da última manutenção (vazia em toda a extração atual).",
    "id_medidor_anterior": "Medidor que este substituiu na mesma UC; nulo se for o primeiro da UC.",
    "tipo_anterior": "Tecnologia do medidor substituído; nula se for o primeiro da UC.",
    "source_file": "Arquivo de origem na landing.",
    "ingestion_datetime": "Momento da carga na Bronze.",
    "silver_datetime": "Momento da carga na Silver.",
}
COLUNAS_SILVER = list(COMENTARIOS)
COMENTARIO_TABELA = (
    "Silver: parque de medidores padronizado e tipado, um registro por medidor (ativos e retirados), "
    "com o medidor anterior de cada UC. Regras em docs/silver/silver_medidores.md."
)

df_final = df_tratado.select(*COLUNAS_SILVER)
for coluna, comentario in COMENTARIOS.items():
    df_final = df_final.withMetadata(coluna, {"comment": comentario})

(
    df_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

if spark.sql(f"DESCRIBE DETAIL {silver_table}").first()["description"] != COMENTARIO_TABELA:
    spark.sql(f"COMMENT ON TABLE {silver_table} IS '{COMENTARIO_TABELA}'")

In [ ]:
# Depois de gravar: a tabela tem as colunas esperadas e o volume validado acima
tabela = spark.table(silver_table)
linhas_tabela = tabela.count()

assert tabela.columns == COLUNAS_SILVER, f"Colunas inesperadas: {tabela.columns}"
assert linhas_tabela == linhas_silver, f"Tabela com {linhas_tabela} linhas, esperado {linhas_silver}"
print(f"OK: {linhas_tabela} medidores carregados em {silver_table}")

## Evidências: antes × depois
As consultas abaixo comparam a Bronze com a Silver. Elas não alteram dados.

In [ ]:
b, s = bronze_table, silver_table

# 1. tipo: grafias da Bronze → domínio da Silver
display(spark.sql(f"""
  SELECT b.tipo AS tipo_bronze, s.tipo AS tipo_silver, COUNT(DISTINCT s.id_medidor) AS medidores
  FROM {b} b JOIN {s} s ON trim(b.id_medidor) = s.id_medidor
  GROUP BY 1, 2 ORDER BY 2, 3 DESC"""))

# 2. Datas: um exemplo de cada formato da Bronze e a data convertida
display(spark.sql(f"""
  SELECT formato, exemplo_bronze, data_silver, linhas FROM (
    SELECT regexp_replace(b.data_instalacao, '[0-9]', '9') AS formato,
           b.data_instalacao AS exemplo_bronze, s.data_instalacao AS data_silver,
           COUNT(*) OVER (PARTITION BY regexp_replace(b.data_instalacao, '[0-9]', '9')) AS linhas,
           ROW_NUMBER() OVER (PARTITION BY regexp_replace(b.data_instalacao, '[0-9]', '9') ORDER BY b.id_medidor) AS n
    FROM {b} b JOIN {s} s ON trim(b.id_medidor) = s.id_medidor)
  WHERE n = 1 ORDER BY linhas DESC"""))

# 3. Visão geral da Silver
display(spark.sql(f"""
  SELECT status, tipo, COUNT(*) AS medidores, MIN(data_instalacao) AS primeira_instalacao,
         MAX(data_instalacao) AS ultima_instalacao, COUNT(data_retirada) AS com_data_retirada
  FROM {s} GROUP BY 1, 2 ORDER BY 1, 2"""))

# 4. Campanha de troca: mês da troca × tecnologia substituída
display(spark.sql(f"""
  SELECT date_format(data_instalacao, 'yyyy-MM') AS mes_troca, tipo_anterior, tipo AS tipo_novo, COUNT(*) AS trocas
  FROM {s} WHERE id_medidor_anterior IS NOT NULL
  GROUP BY 1, 2, 3 ORDER BY 1, 2"""))